# Chapter 6 — Fine-tuning for classification

Chapter 5 produced a reusable GPT architecture and loaded pretrained OpenAI weights. This chapter adapts those representations to a
binary supervised task: classifying SMS messages as legitimate (`ham`) or unwanted (`spam`).

For a modern instruction-tuned LLM, prompting should normally be evaluated before task-specific fine-tuning. This GPT-2-sized model is
not instruction-tuned, however, and the chapter's main purpose is pedagogical: learning dataset preparation, classification heads,
parameter freezing, supervised loss, and evaluation. A production spam system should still be compared with simpler baselines such as
TF–IDF plus logistic regression.

## 6.1 Downloading the SMS Spam Collection

The UCI dataset contains 5,572 labeled SMS messages in a small ZIP archive. Keep acquisition idempotent: if the final TSV already exists,
skip network and extraction work so rerunning the notebook does not overwrite local data.

In [1]:
import os
import urllib.request
import zipfile
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(
    url: str,
    zip_path: str | Path,
    extracted_path: str | Path,
    data_file_path: Path,
) -> None:
    """Download and extract the UCI SMS Spam Collection when absent.

    Args:
        url: URL of the source ZIP archive.
        zip_path: Local path used for the downloaded archive.
        extracted_path: Directory that receives extracted archive members.
        data_file_path: Final path of the renamed tab-separated dataset.

    Returns:
        None.

    Raises:
        urllib.error.URLError: If the dataset download fails.
        zipfile.BadZipFile: If the downloaded archive is invalid.
        OSError: If writing, extracting, or renaming a file fails.
    """
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # The archive is small enough to download into memory before writing.
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # The URL is a trusted UCI source; extract its members into one directory.
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Give the extensionless source file a .tsv suffix that describes its format.
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")


download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping download and extraction.


## 6.2 Loading messages into a dataframe

The source is tab-separated and has no header. Assign `Label` to the external `ham`/`spam` category and `Text` to the raw message. Each
row is one supervised example; the two columns are its target and input.

In [2]:
import pandas as pd

# df: (num_messages=5_572, num_columns=2)
# The source file has no header, so assign descriptive column names.
df = pd.read_csv(
    data_file_path,
    sep="	",
    header=None,
    names=["Label", "Text"],
)
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


### Inspecting class balance

The original dataset contains many more ham messages than spam messages. A classifier that favors ham could therefore obtain deceptively
high raw accuracy without learning useful spam detection.

In [10]:
# Count examples before balancing to expose the original class imbalance.
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


## 6.3 Creating a balanced pedagogical dataset

Keep all 747 spam examples and randomly select 747 ham examples. The fixed seed makes the selected subset reproducible.

Downsampling makes accuracy easier to interpret and reduces training cost, but discards most legitimate messages and changes class
prevalence. It is a teaching simplification rather than a universally preferred production strategy.

In [4]:
def create_balanced_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Downsample ham messages to match the number of spam messages.

    Args:
        df: SMS examples with `Label` and `Text` columns.

    Returns:
        Dataframe containing every spam example and an equally sized,
        reproducibly sampled ham subset.
    """
    # num_spam is the minority-class row count: 747 for this dataset.
    num_spam = df[df["Label"] == "spam"].shape[0]
    # Keep a reproducible subset instead of allowing the majority class to dominate.
    ham_subset = df[df["Label"] == "ham"].sample(
        num_spam,
        random_state=123,
    )
    # balanced_df: (2 * num_spam, num_columns=2)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


### Encoding class labels

Cross-entropy expects integer class targets rather than strings. Map legitimate messages to `0` and spam messages to `1`; these meanings
must remain consistent through datasets, training, evaluation, and inference.

In [5]:
# Convert external string labels into integer targets for cross-entropy.
# ham -> 0 and spam -> 1; balanced_df remains (num_messages=1_494, 2).
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

## 6.4 Creating training, validation, and test splits

Shuffle once with a fixed seed, then allocate 70% for parameter updates, 10% for model-selection feedback, and the remaining 20% for a
final held-out estimate. The test set must not guide training decisions.

Integer boundaries can cause a one-row rounding difference from the exact percentages. All rows still belong to exactly one split.

In [6]:
def random_split(
    df: pd.DataFrame,
    train_frac: float,
    validation_frac: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Shuffle and partition examples into training, validation, and test sets.

    Args:
        df: Labeled examples to partition.
        train_frac: Fraction assigned to the training set.
        validation_frac: Fraction assigned to the validation set. The remaining
            fraction becomes the test set.

    Returns:
        Training, validation, and test dataframes in that order.
    """
    # Shuffle reproducibly so the concatenated classes are mixed before slicing.
    shuffled_df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(shuffled_df) * train_frac)
    validation_end = train_end + int(len(shuffled_df) * validation_frac)

    # With 70%/10%, the unspecificed final 20% becomes the held-out test set.
    train_df = shuffled_df[:train_end]
    validation_df = shuffled_df[train_end:validation_end]
    test_df = shuffled_df[validation_end:]

    return train_df, validation_df, test_df


train_df, validation_df, test_df = random_split(
    balanced_df,
    train_frac=0.7,
    validation_frac=0.1,
)
train_df.shape, validation_df.shape, test_df.shape

((1045, 2), (149, 2), (300, 2))

### Saving reproducible split files

Write each dataframe to a separate CSV. Omitting the pandas index prevents an unrelated numeric column from becoming accidental model
input when the files are loaded later.

In [7]:
# Persist each split without a redundant dataframe-index column.
train_df.to_csv("train.csv", index=False)
validation_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

## 6.5 Reusing the GPT-2 tokenizer

Classification inputs must use the same token-to-ID mapping as pretraining. GPT-2's `<|endoftext|>` marker has a dedicated vocabulary
ID when explicitly permitted through `allowed_special`.

The next dataset stage can use this known token as a separator or padding value without expanding `vocab_size`. Encoding returns a Python
list containing one token ID; tensors and batches will be constructed later.

In [8]:
import tiktoken

# Reuse the tokenizer whose vocabulary matches the pretrained GPT-2 weights.
tokenizer = tiktoken.get_encoding("gpt2")
# One special end-of-text marker encodes as (num_tokens=1,).
endoftext_id = tokenizer.encode(
    "<|endoftext|>",
    allowed_special={"<|endoftext|>"},
)
print(endoftext_id)

[50256]


## Chapter 6 summary

The chapter begins with a reproducible supervised dataset pipeline:

- download and extract the UCI SMS Spam Collection only when needed;
- inspect the natural ham/spam imbalance;
- downsample ham messages to create a balanced pedagogical dataset;
- encode ham as `0` and spam as `1`;
- shuffle before making 70%/10%/20% training, validation, and test splits; and
- persist those splits without dataframe indices; and
- reuse the pretrained GPT-2 tokenizer and its end-of-text vocabulary entry.

Balancing simplifies accuracy interpretation but no longer reflects real-world spam prevalence, so production evaluation would retain a
representative test distribution and compare the LLM with simpler classification baselines.

In [ ]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        self.encoded_texts = [tokenizer.encode(text) for text in self.data["Text"]]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

            self.encoded_texts = [
                encoded_text[: self.max_length] for encoded_text in self.encoded_texts
            ]

        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.long))

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length